# 4. Performance Tracking

## Overview
This notebook implements performance tracking for our ETF portfolio across all four asset classes, including:
1. Historical price data collection via yfinance
2. Return calculations (UK financial year Apr-Mar, YTD, MTD)
3. Risk metrics (volatility, Sharpe ratio)
4. Rebalancing-aware time-weighted return (TWR) from trading statements
5. Portfolio-level P&L and performance reporting

Portfolio data is loaded from the SQLite database (year = 2025), falling back to CSV.

## Required Libraries

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("..").resolve()))

from datetime import datetime, timedelta

import pandas as pd
import numpy as np

from etf_utils.config import DATA_RAW, DATA_INTERMEDIATE, DATA_OUTPUT, DATA_CONFIG
from etf_utils.data_provider import DataProvider
from etf_utils.database import load_portfolio
from etf_utils.metrics import (
    calculate_annualized_volatility,
    calculate_period_metrics,
    calculate_daily_pnl,
)

provider = DataProvider()

# Load portfolio from DB (year=2025)
portfolio_df = load_portfolio(year=2025)
print(f"Loaded {len(portfolio_df)} positions from 2025 portfolio "
      f"(asset classes: {portfolio_df['asset_class'].unique().tolist()})")

## ETF Performance Calculator
Using your existing code to calculate ETF metrics:

In [ ]:
def calculate_etf_metrics(symbol, fy_start_years=range(2020, 2026)):
    """Calculate UK financial year (Apr-Mar) performance metrics for an ETF using DataProvider."""
    df = provider.get_historical_prices(symbol)
    annual_results = []

    for start_year in fy_start_years:
        fy_start = pd.Timestamp(f"{start_year}-04-01")
        fy_end = pd.Timestamp(f"{start_year + 1}-03-31")

        df_fy = df[(df.index >= fy_start) & (df.index <= fy_end)]
        if len(df_fy) < 2:
            continue

        start_price = df_fy["close"].iloc[0]
        end_price = df_fy["close"].iloc[-1]
        annual_return = ((end_price / start_price) - 1) * 100
        volatility = calculate_annualized_volatility(df_fy["close"]) * 100

        end_yr = str(start_year + 1)[2:]
        if fy_end > pd.Timestamp.now():
            label = f"Apr '{str(start_year)[2:]} - to date"
        else:
            label = f"Apr '{str(start_year)[2:]} - Mar '{end_yr}"

        annual_results.append({
            "_fy_start": start_year,
            "Financial Year": label,
            "Start Price": round(float(start_price), 2),
            "End Price": round(float(end_price), 2),
            "Annualized Return (%)": round(float(annual_return), 2),
            "Volatility (%)": round(float(volatility), 2),
        })

    result = pd.DataFrame(annual_results)
    if not result.empty:
        result = result.sort_values("_fy_start").reset_index(drop=True)
    return result


## Annual Performance Metrics

## Portfolio Performance
Calculate weighted portfolio performance:

## Calculate Portfolio Performance
First, we'll calculate the performance metrics for each ETF and combine them into portfolio-level metrics:

In [ ]:
# Initialize portfolio performance tracking
portfolio_performance = pd.DataFrame()

# Calculate performance for each ETF using DataProvider (no API key needed)
for idx, row in portfolio_df.iterrows():
    try:
        etf_metrics = calculate_etf_metrics(row['ticker'])
        if not etf_metrics.empty:
            weight = row['investment'] / portfolio_df['investment'].sum()
            etf_metrics['Weighted Return (%)'] = etf_metrics['Annualized Return (%)'] * weight
            etf_metrics['Weighted Volatility (%)'] = etf_metrics['Volatility (%)'] * weight
            etf_metrics['ETF'] = row['ticker']
            portfolio_performance = pd.concat([portfolio_performance, etf_metrics])
    except Exception as e:
        print(f"Error processing {row['ticker']}: {str(e)}")

# Calculate portfolio-level metrics
if not portfolio_performance.empty:
    portfolio_summary = portfolio_performance.groupby(['_fy_start', 'Financial Year']).agg({
        'Weighted Return (%)': 'sum',
        'Weighted Volatility (%)': 'sum',
    }).reset_index().sort_values('_fy_start').reset_index(drop=True)
    portfolio_summary = portfolio_summary.rename(columns={
        'Weighted Return (%)': 'Return (%)',
        'Weighted Volatility (%)': 'Volatility (%)',
    }).drop(columns='_fy_start')
    print("\nPortfolio Performance Summary:")
    display(portfolio_summary)
else:
    print("No performance data available")

In [ ]:
fy25 = portfolio_performance[portfolio_performance['_fy_start'] == 2025]
display(fy25[['ETF', 'Financial Year', 'Start Price', 'End Price', 'Annualized Return (%)', 'Volatility (%)']])


In [ ]:
# Calculate risk-adjusted metrics from portfolio summary
from etf_utils.metrics import calculate_sharpe_ratio

if 'portfolio_summary' in locals() and not portfolio_summary.empty:
    risk_free_rate = 4.0  # UK base rate ~4% in 2024
    portfolio_summary['Sharpe Ratio'] = portfolio_summary.apply(
        lambda row: round(calculate_sharpe_ratio(
            row['Return (%)'], row['Volatility (%)'], risk_free_rate
        ), 2),
        axis=1,
    )
    print("\nRisk-Adjusted Performance Metrics:")
    display(portfolio_summary)
else:
    print("Unable to calculate risk metrics - no performance data available")

## Recent Performance Metrics (YTD and MTD)
Calculate performance metrics for:
1. Year to Date (2025)
2. Month to Date

In [ ]:
# Calculate YTD and MTD performance using DataProvider
ytd_start = "2025-01-01"
mtd_start = "2025-02-01"

ytd_performance = []
mtd_performance = []

for idx, row in portfolio_df.iterrows():
    try:
        df = provider.get_historical_prices(row['ticker'])
        weight = row['investment'] / portfolio_df['investment'].sum()

        # YTD metrics
        ytd = calculate_period_metrics(df, ytd_start)
        if ytd and not np.isnan(ytd['return']):
            ytd_performance.append({
                'ETF': row['ticker'],
                'Return (%)': round(ytd['return'] * 100, 2),
                'Volatility (%)': round(ytd['volatility'] * 100, 2),
                'Weight': weight,
                'Weighted Return (%)': round(ytd['return'] * 100 * weight, 2),
                'Weighted Volatility (%)': round(ytd['volatility'] * 100 * weight, 2),
            })

        # MTD metrics
        mtd = calculate_period_metrics(df, mtd_start)
        if mtd and not np.isnan(mtd['return']):
            mtd_performance.append({
                'ETF': row['ticker'],
                'Return (%)': round(mtd['return'] * 100, 2),
                'Volatility (%)': round(mtd['volatility'] * 100, 2),
                'Weight': weight,
                'Weighted Return (%)': round(mtd['return'] * 100 * weight, 2),
                'Weighted Volatility (%)': round(mtd['volatility'] * 100 * weight, 2),
            })
    except Exception as e:
        print(f"Error processing {row['ticker']}: {str(e)}")

ytd_df = pd.DataFrame(ytd_performance)
mtd_df = pd.DataFrame(mtd_performance)

if not ytd_df.empty:
    print("\nYear to Date (2025) Portfolio Performance:")
    print(f"Total Return: {ytd_df['Weighted Return (%)'].sum():.2f}%")
    print(f"Portfolio Volatility: {ytd_df['Weighted Volatility (%)'].sum():.2f}%")
    print("\nTop 3 YTD Contributors:")
    display(ytd_df.nlargest(3, 'Weighted Return (%)')[['ETF', 'Return (%)', 'Weighted Return (%)']])

if not mtd_df.empty:
    print("\nMonth to Date Portfolio Performance:")
    print(f"Total Return: {mtd_df['Weighted Return (%)'].sum():.2f}%")
    print(f"Portfolio Volatility: {mtd_df['Weighted Volatility (%)'].sum():.2f}%")

# YTD Sharpe ratio
risk_free_rate = 0.04
if not ytd_df.empty:
    ytd_return = ytd_df['Weighted Return (%)'].sum() / 100
    ytd_vol = ytd_df['Weighted Volatility (%)'].sum() / 100
    ytd_sharpe = (ytd_return - risk_free_rate * 0.5) / ytd_vol if ytd_vol > 0 else float('nan')
    print(f"\nYTD Sharpe Ratio: {ytd_sharpe:.2f}")

## Rebalancing-Aware Portfolio Performance (Time-Weighted Return)

This section parses the InvestEngine trading statement to calculate a proper
time-weighted return that accounts for rebalancing events.

In [ ]:
"""
Rebalancing-Aware Portfolio Performance (Time-Weighted Return)
==============================================================
Paste this into a new cell in 04_performance_tracking.ipynb,
after the existing portfolio performance cells.

Requires: pd, np, Path, DATA_OUTPUT already defined.
"""

import re
import importlib
from collections import defaultdict
import pandas as pd
import numpy as np
from pathlib import Path

# Reload data_provider to pick up the updated multi-jump GBX fix
import etf_utils.data_provider
importlib.reload(etf_utils.data_provider)
from etf_utils.data_provider import DataProvider
from etf_utils.database import save_rebalancing_trades, load_rebalancing_trades
provider = DataProvider()  # Fresh instance with updated code

# ---------- 1. ISIN → Ticker mapping ----------
ISIN_TO_TICKER = {
    "IE00BZ163G84": "VECP",   # Vanguard EUR Corporate Bond
    "IE00B1XNH568": "IMIB",   # iShares FTSE MIB
    "IE00B0M63516": "IBZL",   # iShares MSCI Brazil
    "LU1048314949": "UC81",   # UBS US Liquid Corp Bonds
    "LU1931975152": "PRIR",   # Amundi Euro Govt Bonds
    "IE00B6TLBW47": "EMCP",   # iShares EM Corp Bond
    "IE00B00FV011": "SLXX",   # iShares Corporate Bond
    "IE00BD4TY345": "AUAD",   # UBS MSCI Australia
    "IE00B1FZSB30": "IGLT",   # iShares Core UK Gilts
    "IE00BF2FN646": "TRXG",   # Invesco US Treasury 7-10Y
    "IE00B44T3H88": "HMCH",   # HSBC MSCI China
    "LU0838782315": "XDDX",   # Xtrackers DAX
    "IE00BZ163L38": "VEMT",   # Vanguard EM Govt Bonds
    "LU1931974775": "PRIJ",   # Amundi Prime Japan
    "IE00BM8R0J59": "QYLP",   # Global X Nasdaq 100 CC
    "LU1781541096": "LCUK",   # Amundi Core UK Equity
}

from etf_utils.config import PROJECT_ROOT

# ---------- 2. Load trades: CSV (parse + save to DB) or DB fallback ----------
PORTFOLIO_YEAR = 2025

statement_path = (
    PROJECT_ROOT
    / "data"
    / "investment_statements"
    / f"RebalancingTrades{PORTFOLIO_YEAR}.csv"
)

if statement_path.exists():
    # CSV exists: parse and persist to DB
    trades_raw = pd.read_csv(statement_path, skiprows=2, header=None,
                             names=["security", "type", "quantity", "price", "value",
                                    "trade_datetime", "settlement_date", "broker"])

    trades_raw["isin"] = trades_raw["security"].str.extract(r"ISIN\s+([A-Z]{2}[A-Z0-9]{10})")
    trades_raw["ticker"] = trades_raw["isin"].map(ISIN_TO_TICKER)

    trades_raw["trade_date"] = pd.to_datetime(
        trades_raw["trade_datetime"].str.strip(), format="%d/%m/%y %H:%M:%S"
    ).dt.normalize()

    trades_raw["quantity"] = pd.to_numeric(trades_raw["quantity"], errors="coerce")
    # Strip currency symbol (£ / \u00a3) and commas from price/value columns
    trades_raw["price"] = (
        trades_raw["price"].astype(str).str.replace("\u00a3", "", regex=False).str.replace(",", "")
    )
    trades_raw["price"] = pd.to_numeric(trades_raw["price"], errors="coerce")
    trades_raw["value"] = (
        trades_raw["value"].astype(str).str.replace("\u00a3", "", regex=False).str.replace(",", "")
    )
    trades_raw["value"] = pd.to_numeric(trades_raw["value"], errors="coerce")

    trades = trades_raw.dropna(subset=["ticker", "trade_date"]).copy()

    trades["signed_qty"] = trades.apply(
        lambda r: r["quantity"] if r["type"].strip() == "Buy" else -r["quantity"], axis=1
    )
    trades["signed_value"] = trades.apply(
        lambda r: r["value"] if r["type"].strip() == "Buy" else -r["value"], axis=1
    )

    save_rebalancing_trades(trades, portfolio_year=PORTFOLIO_YEAR)
    print(f"Parsed CSV and saved {len(trades)} trades to DB for year {PORTFOLIO_YEAR}")
else:
    # No CSV: load from DB
    trades = load_rebalancing_trades(portfolio_year=PORTFOLIO_YEAR)
    if trades.empty:
        raise FileNotFoundError(
            f"No trading statement CSV at {statement_path} and no trades in DB "
            f"for portfolio year {PORTFOLIO_YEAR}"
        )
    # trade_date round-trips as TEXT in SQLite — restore to datetime
    trades["trade_date"] = pd.to_datetime(trades["trade_date"])
    print(f"Loaded {len(trades)} trades from DB for year {PORTFOLIO_YEAR}")

print(f"Loaded {len(trades)} transactions for {trades['ticker'].nunique()} tickers")
print(f"Date range: {trades['trade_date'].min().date()} to {trades['trade_date'].max().date()}")

rebalance_dates = sorted(trades["trade_date"].unique())
print(f"{len(rebalance_dates)} rebalancing dates identified")

# ---------- 3. Fetch price data ----------
all_tickers = sorted(trades["ticker"].unique().tolist())

price_data = {}
for ticker in all_tickers:
    try:
        price_data[ticker] = provider.get_historical_prices(ticker)
    except Exception as e:
        print(f"  Warning: could not fetch data for {ticker}: {e}")

# ---------- 4. DIAGNOSTIC: Per-ETF return check ----------
# Compare yfinance returns vs trade statement prices for each ticker
first_trade_date = trades["trade_date"].min()
today = pd.Timestamp.now().normalize()

print("\n" + "=" * 70)
print("DIAGNOSTIC: Per-ETF yfinance returns from first trade to today")
print("=" * 70)
print(f"{'Ticker':<8} {'Trade Price':>12} {'YF Start':>12} {'YF End':>12} {'YF Return':>10} {'Weight':>8}")
print("-" * 70)

# Get first day trades for initial values
first_day_trades = trades[trades["trade_date"] == first_trade_date]
etf_initial_values = {}
for _, trade in first_day_trades.iterrows():
    etf_initial_values[trade["ticker"]] = etf_initial_values.get(trade["ticker"], 0) + trade["signed_value"]

total_initial = sum(v for v in etf_initial_values.values() if v > 0)

for ticker in all_tickers:
    if ticker not in price_data:
        print(f"{ticker:<8} NO DATA")
        continue

    df = price_data[ticker]

    # First trade price
    ticker_trades = trades[trades["ticker"] == ticker]
    trade_price = ticker_trades.iloc[0]["price"]

    # yfinance price near first trade
    after_start = df[df.index >= first_trade_date]
    before_end = df[df.index <= today]

    yf_start = float(after_start["close"].iloc[0]) if not after_start.empty else 0
    yf_end = float(before_end["close"].iloc[-1]) if not before_end.empty else 0

    yf_return = ((yf_end / yf_start) - 1) * 100 if yf_start > 0 else 0

    weight = etf_initial_values.get(ticker, 0) / total_initial * 100 if total_initial > 0 else 0

    print(f"{ticker:<8} \u00a3{trade_price:>10.2f} {yf_start:>12.2f} {yf_end:>12.2f} {yf_return:>9.2f}% {weight:>7.1f}%")

# ---------- 5. Build portfolio & calculate TWR ----------
etf_values = defaultdict(float)
for _, trade in first_day_trades.iterrows():
    etf_values[trade["ticker"]] += trade["signed_value"]

initial_portfolio_value = sum(etf_values.values())
print(f"\nInitial portfolio value: \u00a3{initial_portfolio_value:,.2f}")

remaining_rebalance_dates = [d for d in rebalance_dates if d > first_trade_date]
period_boundaries = sorted(set([first_trade_date] + remaining_rebalance_dates + [today]))

sub_period_returns = []


def get_yf_return(ticker, date_start, date_end):
    """Get return ratio from yfinance (unit-agnostic)."""
    if ticker not in price_data:
        return 0.0
    df = price_data[ticker]
    after_start = df[df.index >= date_start]
    before_end = df[df.index <= date_end]
    if after_start.empty or before_end.empty:
        return 0.0
    
    # If there's a massive gap that completely skips this period,
    # the first available date after start is LATER than the last available date before end.
    if after_start.index[0] >= before_end.index[-1]:
        return 0.0
        
    p_start = float(after_start["close"].iloc[0])
    p_end = float(before_end["close"].iloc[-1])
    if p_start == 0:
        return 0.0
    return (p_end / p_start) - 1


for i in range(len(period_boundaries) - 1):
    p_start = period_boundaries[i]
    p_end = period_boundaries[i + 1]

    if p_start >= p_end:
        continue

    total_value = sum(v for v in etf_values.values() if v > 0)
    if total_value <= 0:
        continue

    port_return = 0.0
    for ticker, value in etf_values.items():
        if value <= 0:
            continue
        weight = value / total_value
        etf_return = get_yf_return(ticker, p_start, p_end)
        port_return += weight * etf_return

    # Apply the actual price change to existing holdings
    for ticker in list(etf_values.keys()):
        if etf_values[ticker] > 0:
            etf_return = get_yf_return(ticker, p_start, p_end)
            etf_values[ticker] *= (1 + etf_return)

    # Collect rebalancing trades for this exact date
    day_trades = trades[trades["trade_date"] == p_end]
    
    # Apply "Buy/Sell" trades
    for _, trade in day_trades.iterrows():
        etf_values[trade["ticker"]] += trade["signed_value"]  

    port_value_end = sum(v for v in etf_values.values() if v > 0)
    
    true_sub_period_return = (port_value_end - total_value) / total_value if total_value > 0 else 0.0

    sub_period_returns.append({
        "period_start": p_start,
        "period_end": p_end,
        "days": (p_end - p_start).days,
        "portfolio_value_start": round(total_value, 2),
        "portfolio_value_end": round(port_value_end, 2),
        "sub_period_return": true_sub_period_return,
    })

# ---------- 6. Chain TWR ----------
cumulative_twr = 1.0
for sp in sub_period_returns:
    cumulative_twr *= (1 + sp["sub_period_return"])
    sp["cumulative_return"] = (cumulative_twr - 1) * 100

total_twr = (cumulative_twr - 1) * 100

# ---------- 7. Display ----------
print("\n" + "=" * 60)
print("=" * 60)
print(f"\nPeriod: {first_trade_date.date()} to {today.date()}")
print(f"Rebalancing events: {len(remaining_rebalance_dates)}")
print(f"\nTotal Time-Weighted Return: {total_twr:.2f}%")

if sub_period_returns:
    initial_value = sub_period_returns[0]["portfolio_value_start"]
    final_value = sub_period_returns[-1]["portfolio_value_end"]
    print(f"Portfolio value: \u00a3{initial_value:,.2f} to \u00a3{final_value:,.2f}")

    # Sub-period detail table
    print("\n--- Sub-Period Detail ---")
    sp_df = pd.DataFrame(sub_period_returns)
    sp_df["period_start"] = sp_df["period_start"].dt.date
    sp_df["period_end"] = sp_df["period_end"].dt.date
    sp_df["sub_period_return"] = (sp_df["sub_period_return"] * 100).round(2)
    sp_df["cumulative_return"] = sp_df["cumulative_return"].round(2)
    print(sp_df.to_string())

    # Monthly summary
    monthly_returns = defaultdict(lambda: 1.0)
    for sp in sub_period_returns:
        month_key = sp["period_start"].strftime("%b '%y") if isinstance(sp["period_start"], pd.Timestamp) else str(sp["period_start"])[:7]
        monthly_returns[month_key] *= (1 + sp["sub_period_return"] / 100)

    monthly_summary = [{"Month": m, "Return (%)": round((c - 1) * 100, 2)} for m, c in monthly_returns.items()]
    print("\n--- Monthly Breakdown ---")
    print(pd.DataFrame(monthly_summary).to_string())

    # Current holdings
    print("\n--- Current Holdings ---")
    holdings_rows = [{"Ticker": t, "Value (\u00a3)": round(etf_values[t], 2)} for t in sorted(etf_values.keys()) if etf_values[t] > 0]
    holdings_df = pd.DataFrame(holdings_rows)
    print(holdings_df.to_string())
    print(f"Total: \u00a3{holdings_df['Value (\u00a3)'].sum():,.2f}")
else:
    print("No sub-period returns could be calculated (check trade data parsing)")

## Daily PnL Tracking

In [ ]:
# Calculate daily PnL for the portfolio using DataProvider
pnl_start = "2025-02-01"

daily_pnl_list = []

for idx, row in portfolio_df.iterrows():
    try:
        df = provider.get_historical_prices(row['ticker'])
        pnl_df = calculate_daily_pnl(df, row['investment'], pnl_start)

        if pnl_df is not None and not pnl_df.empty:
            weight = row['investment'] / portfolio_df['investment'].sum()
            pnl_df = pnl_df.copy()
            pnl_df['ETF'] = row['ticker']
            pnl_df['Weight'] = weight
            pnl_df['Weighted PnL'] = pnl_df['pnl'] * weight
            daily_pnl_list.append(pnl_df)
    except Exception as e:
        print(f"Error processing {row['ticker']}: {str(e)}")

if daily_pnl_list:
    daily_pnl = pd.concat(daily_pnl_list)
    print("\nDaily PnL Summary by ETF:")
    display(daily_pnl.groupby('ETF')['Weighted PnL'].sum().sort_values(ascending=False))

    # Portfolio-level daily PnL
    portfolio_daily = daily_pnl.groupby(daily_pnl.index)['Weighted PnL'].sum()
    print(f"\nTotal Portfolio PnL since {pnl_start}: {portfolio_daily.sum():.2f}")
else:
    print("No PnL data available")

In [ ]:
import plotly.graph_objects as go

# Plot daily returns for a selected ETF
selected_etf = "AUAD"
etf_data = daily_pnl[daily_pnl['ETF'] == selected_etf].copy()

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=etf_data.index,
        y=etf_data['daily_return'],
        mode='lines+markers',
        name='Daily Return',
        line=dict(color='blue'),
        hovertemplate='Date: %{x}<br>Return: %{y:.2%}<extra></extra>'
    )
)

fig.update_layout(
    title=f'{selected_etf} ETF Daily Returns',
    xaxis_title='Date',
    yaxis_title='Daily Return',
    yaxis_tickformat='.2%',
    hovermode='x unified',
    template='plotly_white',
    width=1000,
    height=600
)

fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig.show()

In [ ]:
# Show recent data for selected ETF
etf_data.tail(10)

In [ ]:
# Portfolio-level daily PnL
portfolio_daily_pnl = daily_pnl.groupby(daily_pnl.index)['Weighted PnL'].sum()
portfolio_daily_pnl.tail(10)